*This notebook contains exploratory development checks. Its quick-mode outputs are not full-study results. The completed five-dimensional comparison used fixed neural settings and did not complete the calibration grid described below.*

# 02 — Neural training diagnostics

This notebook explores neural training settings using development paths separate from the study's test set. It includes learning-rate and step-budget comparisons, a checkpoint diagnostic, runtime measurements and a check of the validation-based exercise guard.

**The completed five-dimensional study did not use a finished tuning grid.** Its neural settings were fixed at 1,500 Adam steps, learning rate 0.001 and batch size 512. Results from the exercises below should not be interpreted as evidence of optimal hyperparameters, successful training saturation or a completed development-cost comparison.

In [ ]:
MODE = "quick"

if MODE != "quick":
    raise RuntimeError("Calibration grid omitted from the frozen study; quick smoke workflow only.")

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stopping import *

print("Thread limits:", limit_threads(1))
BASE = CONFIG[MODE]["base_seed"]
OUT = results_dir(MODE)
print("Mode:", MODE, "| results ->", OUT)
cal = CONFIG[MODE]["calibration"]
cal

## 1. Development grid

Compare neural training configurations and validation-tuned LSM on a separate selection sample. That sample is used for model selection, so its reported values are not unbiased out-of-sample results.

In [ ]:
opt = paper_max_call(5)
ref = reference_for(opt)
selection_set = gbm_paths(opt, cal["selection_set"], seed_for(BASE, "holdout", 5))
cal_path = calibration_path(OUT, MODE)
print("Provenance-checked calibration:", cal_path)
done = pd.read_csv(cal_path) if cal_path.exists() else pd.DataFrame()
def is_done(**key):
    if done.empty: return False
    mask = np.ones(len(done), bool)
    for k, v in key.items(): mask &= (done[k] == v).to_numpy()
    return mask.any()

CAL_COLUMNS = ["method", "variant", "role", "budget", "dev_seed", "lr", "max_steps", "selection_value",
               "selection_unguarded", "n_guarded", "at_limit_share", "fit_s", "row_wall_s"]
def save(row):
    pd.DataFrame([row])[CAL_COLUMNS].to_csv(cal_path, mode="a", header=not cal_path.exists(), index=False)

all_variants = cal["variants"] + cal["diagnostic_variants"]
wall0 = time.perf_counter()
for budget in cal["budgets"]:
    for dev_seed in cal["dev_seeds"]:
        t_row = time.perf_counter()
        train = gbm_paths(opt, budget, seed_for(BASE, "train", 5, dev_seed))
        val = gbm_paths(opt, cal["val"], seed_for(BASE, "val", 5, dev_seed))
        if not is_done(method="lsm", budget=budget, dev_seed=dev_seed):
            lsm, _ = select_lsm(opt, train, val)
            save(dict(method="lsm", variant="lsm", role="reference", budget=budget, dev_seed=dev_seed, lr=np.nan, max_steps=np.nan,
                      selection_value=discounted_payoffs(opt, selection_set, lsm)[0].mean(),
                      selection_unguarded=discounted_payoffs(opt, selection_set, lsm, False)[0].mean(),
                      n_guarded=lsm.n_guarded(), at_limit_share=np.nan, fit_s=lsm.info["selection_s"],
                      row_wall_s=time.perf_counter() - t_row))
        for variant in all_variants:
            for lr in cal["lrs"]:
                for steps in cal["max_steps"]:
                    key = dict(method="nn", variant=variant, budget=budget, dev_seed=dev_seed, lr=lr, max_steps=steps)
                    if is_done(**key):
                        continue
                    t_row = time.perf_counter()
                    s = with_variant(NeuralSettings(max_steps=steps, lr=lr, batch=cal["batch"]), variant)
                    pol = fit_neural(opt, train, val, s, seed=seed_for(BASE, "init", 5, dev_seed))
                    best = [r["best_step"] for r in pol.diagnostics if r["best_step"] > 0]
                    save(dict(**key, role="selection" if variant in cal["variants"] else "diagnostic",
                              selection_value=discounted_payoffs(opt, selection_set, pol)[0].mean(),
                              selection_unguarded=discounted_payoffs(opt, selection_set, pol, False)[0].mean(),
                              n_guarded=pol.n_guarded(),
                              at_limit_share=np.mean([b >= 0.9 * steps for b in best]) if best else np.nan,
                              fit_s=pol.info["train_s"], row_wall_s=time.perf_counter() - t_row))
        print(f"budget {budget:>6}, dev seed {dev_seed}: done  ({(time.perf_counter() - wall0) / 60:.1f} min this session)")

cal_df = pd.read_csv(cal_path)
cal_df["gap"] = ref["value"] - cal_df.selection_value
print(f"Development cost: {cal_df.fit_s.sum() / 60:.1f} CPU-min model fitting; "
      f"{cal_df.row_wall_s.sum() / 60:.1f} min total elapsed (incl. simulation and evaluation)")

## 2. Training-step sensitivity

Plot validation-set values against optimisation steps and compare the observed improvement across development seeds. The share of checkpoints selected near the last training steps is a diagnostic of whether the step budget may be too short, not a convergence test.

In [ ]:
nn_df = cal_df[cal_df.method == "nn"]
fig, axes = plt.subplots(1, len(cal["budgets"]), figsize=(6 * len(cal["budgets"]), 4), squeeze=False)
for ax, budget in zip(axes[0], cal["budgets"]):
    lsm_vals = cal_df[(cal_df.method == "lsm") & (cal_df.budget == budget)].selection_value
    m, _, lo, hi = mean_ci(lsm_vals)
    if np.isfinite(lo):
        ax.axhspan(lo, hi, color="black", alpha=0.08)
    ax.axhline(m, color="black", ls="--", label="tuned LSM (dev seeds, mean ± CI)")
    ax.axhspan(ref["low"], ref["high"], color="grey", alpha=0.25, label="published reference")
    for (variant, lr), sub in nn_df[nn_df.budget == budget].groupby(["variant", "lr"]):
        stats = sub.groupby("max_steps").selection_value.apply(lambda s: pd.Series(mean_ci(s), index=["m", "se", "lo", "hi"])).unstack()
        err = np.vstack([stats.m - stats.lo, stats.hi - stats.m]) if stats.lo.notna().all() else None
        ax.errorbar(stats.index, stats.m, yerr=err, fmt="o:" if variant in cal["diagnostic_variants"] else "o-",
                    capsize=3, label=f"{variant}, lr={lr:g}")
    ax.set(xscale="log", xlabel="Max Adam steps per exercise date", ylabel="Selection-set policy value",
           title=f"5D max-call, {budget:,} training paths")
    ax.set_xticks(cal["max_steps"], [f"{s:,}" for s in cal["max_steps"]]); ax.minorticks_off()
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "calibration_saturation.png", dpi=160); plt.show()
(nn_df.groupby(["budget", "variant", "lr", "max_steps"])
      [["selection_value", "gap", "n_guarded", "at_limit_share", "fit_s"]].mean().round(4))

## 3. Configuration selection in the exploratory workflow

For each learning rate and step budget, compare the mean validation values across development seeds. The code first looks for a configuration within a specified value tolerance of the best candidate in every evaluated case, preferring fewer steps. If no configuration meets that criterion, it chooses the best average value. A separate comparison of the largest step budgets flags possible continued improvement.

This rule is part of the exploratory notebook; **the completed study did not run this calibration experiment**.

In [ ]:
tol = cal["tolerance"]
sel_df = nn_df[nn_df.role == "selection"]
means = sel_df.groupby(["variant", "budget", "lr", "max_steps"]).selection_value.mean().unstack(["variant", "budget"])
best_per_cell = means.max(axis=0)
adequate = (means >= best_per_cell - tol).all(axis=1)
average = means.mean(axis=1)
candidates = average[adequate] if adequate.any() else average
order = sorted(candidates.index, key=lambda c: (c[1] if adequate.any() else 0, -candidates[c]))
lr_star, steps_star = order[0]

steps_sorted = sorted(cal["max_steps"])
sat_rows = []
if len(steps_sorted) >= 2:
    chosen = nn_df[nn_df.lr == lr_star].pivot_table(index=["variant", "budget", "dev_seed"], columns="max_steps", values="selection_value")
    per_seed = (chosen[steps_sorted[-1]] - chosen[steps_sorted[-2]]).rename("improvement").reset_index()
    for (variant, budget), sub in per_seed.groupby(["variant", "budget"]):
        m, se, lo, hi = mean_ci(sub.improvement)
        sat_rows.append(dict(cell=f"{variant}@{budget}", role="selection" if variant in cal["variants"] else "diagnostic",
                             improvement=m, ci_low=lo, ci_high=hi, dev_seeds=len(sub), flagged=bool(m > tol)))
sat_df = pd.DataFrame(sat_rows)
not_saturated = {r.cell: r.improvement for r in sat_df.itertuples() if r.flagged}

frozen = dict(provenance=calibration_provenance(MODE), calibration_file=cal_path.name, neural=asdict(NeuralSettings(max_steps=int(steps_star), lr=float(lr_star), batch=cal["batch"])),
              rule=f"fewest steps within {tol} of the best in every (selection variant, budget) cell",
              selection_variants=cal["variants"], diagnostic_variants=cal["diagnostic_variants"],
              any_adequate=bool(adequate.any()),
              saturation=sat_rows, not_saturated_cells=not_saturated,
              development_cost_cpu_minutes=float(cal_df.fit_s.sum() / 60),
              development_elapsed_minutes=float(cal_df.row_wall_s.sum() / 60),
              selection_set_paths=cal["selection_set"], dev_seeds=cal["dev_seeds"], mode=MODE)
(OUT / "frozen_settings.json").write_text(json.dumps(frozen, indent=2))
print("Frozen neural settings:", frozen["neural"])
print(f"Improvement of {steps_sorted[-1]} over {steps_sorted[-2]} steps (lr={lr_star:g}), across development seeds:")
display(sat_df.round(4))
if not_saturated:
    print("WARNING: training still improves at the largest step limit in", list(not_saturated),
          "\n -> report this. If a SELECTION variant is flagged, consider adding a larger step limit to CONFIG and "
          "rerunning this notebook BEFORE notebook 03. A flagged DIAGNOSTIC variant is a finding (Claim B), not a blocker.")

## 4. Runtime check

An optional development-only calculation measures approximate runtimes for fitting and nested simulation. It is disabled by default. The original full-study runtime estimates and study plan are available in the historical [`study-v1`](https://github.com/MoritzDue/bermudan-neural-stopping/tree/study-v1) version; this cell does not reproduce the full experiment.

In [ ]:
RUN_PREFLIGHT = False
study = CONFIG[MODE]["study"]
pre_path = OUT / "preflight.json"
if RUN_PREFLIGHT:
    settings = NeuralSettings(**frozen["neural"])
    from dual import policy_dual
    unit_s = {}
    timing_train = gbm_paths(opt, max(cal["budgets"]), seed_for(BASE, "train", 5, 998))
    timing_val = gbm_paths(opt, cal["val"], seed_for(BASE, "val", 5, 998))
    calibration_fit_s = {}
    for steps in cal["max_steps"]:
        t0 = time.perf_counter()
        timing_policy = fit_neural(opt, timing_train, timing_val,
            NeuralSettings(max_steps=steps, lr=float(lr_star), batch=cal["batch"]),
            seed=seed_for(BASE, "init", 5, 998))
        calibration_fit_s[str(steps)] = time.perf_counter() - t0
    t0 = time.perf_counter()
    paper_timing = fit_neural(opt, None, timing_val,
        NeuralSettings(sorted_inputs=False, payoff_input=True, fresh_paths=True,
                       **CONFIG[MODE]["validation"]["paper_check"]),
        seed=seed_for(BASE, "check", 5, 998))
    paper_seconds = time.perf_counter() - t0
    dual_chunk_s = {}
    timing_lsm = select_lsm(opt, timing_train, timing_val)[0]
    for method, policy in (("lsm", timing_lsm), ("nn", timing_policy)):
        t0 = time.perf_counter()
        policy_dual(opt, policy, outer=8, inner=128, outer_chunk=8,
                    base_seed=BASE, seed=998, memory_mb=256)
        dual_chunk_s[method] = time.perf_counter() - t0
    for d, budgets in ((5, study["budgets"]), (2, [study["case_2d"]["budget"]])):
        o = paper_max_call(d)
        test_like = gbm_paths(o, study["test"], seed_for(BASE, "holdout", d, extra=1))
        for b in budgets:
            variants = list(VARIANTS) if (d == 5 and b in study["ablation_budgets"]) else [MAIN_VARIANT]
            t0 = time.perf_counter()
            run_unit(o, b, 999, variants, settings, study["val"], test_like, reference_for(o), BASE)
            unit_s[f"{d}d@{b}"] = time.perf_counter() - t0
            print(f"{d}D, {b:>6} paths, {len(variants)} neural variant(s): {unit_s[f'{d}d@{b}']:.1f} s per seed")
    projected = sum(unit_s[f"5d@{b}"] for b in study["budgets"]) * len(study["seeds"]) + \
                unit_s[f"2d@{study['case_2d']['budget']}"] * len(study["case_2d"]["seeds"])
    calibration_projected = (sum(calibration_fit_s.values()) * len(cal["budgets"])
        * len(cal["dev_seeds"]) * len(all_variants) * len(cal["lrs"]))
    dual_policies = (2*len(study["budgets"])*len([s for s in (0, 1, 2) if s in study["seeds"]])
        + 3*len(study["ablation_budgets"]) + 2)
    dual_outer, dual_inner = ((32, 64) if MODE == "quick" else (2000, 3200))
    dual_projected = ((dual_policies + 4) * max(dual_chunk_s.values())
        * dual_outer/8 * dual_inner/128 + (dual_policies + 2)*max(unit_s.values()))
    remaining_seconds = projected + dual_projected
    preflight = dict(provenance=calibration_provenance(MODE), neural=frozen["neural"],
        unit_seconds=unit_s, projected_minutes=projected/60,
        calibration_fit_seconds=calibration_fit_s,
        calibration_projected_minutes=calibration_projected/60,
        paper_check_seconds=paper_seconds, dual_chunk_seconds=dual_chunk_s,
        dual_projection=dict(policies=dual_policies, outer=dual_outer, inner=dual_inner,
                             minutes=dual_projected/60),
        remaining_study_dual_minutes=remaining_seconds/60,
        note="Linear timing projection, not an eight-hour authorization gate. The full run must add actual elapsed validation/calibration/diagnostic cost and reserve for inner sweeps and reproduction. Full calibration precedes this cell; a timing preflight must run before large jobs. No test results used.")
    pre_path.write_text(json.dumps(preflight, indent=2))
if pre_path.exists():
    print(pre_path.read_text())
else:
    print("No preflight yet. Set RUN_PREFLIGHT = True and run this cell when you are ready for the full main study.")

## 5. Validation-guard diagnostic

The exercise guard suppresses a date-specific rule when its estimated validation benefit is negative. Neural checkpoint selection and the guard both use validation information, so the resulting validation gains can be optimistic.

This diagnostic estimates the effect of the original guard decisions on a separate set of paths, without changing fitted rules. Date-level gains are not additive policy values, and the small quick-mode sample is insufficient for firm conclusions.

In [ ]:
guard_rows, guard_policy_rows = [], []
for budget in cal["budgets"]:
    for dev_seed in cal["dev_seeds"]:
        train = gbm_paths(opt, budget, seed_for(BASE, "train", 5, dev_seed))
        val = gbm_paths(opt, cal["val"], seed_for(BASE, "val", 5, dev_seed))
        fresh_seed = seed_for(BASE, 'holdout', 5, dev_seed, extra=74)
        fresh = gbm_paths(opt, cal["selection_set"], fresh_seed)
        policies = {"lsm": select_lsm(opt, train, val)[0]}
        for variant in cal["variants"]:
            policies["nn_" + variant] = fit_neural(
                opt, train, val, with_variant(NeuralSettings(**frozen["neural"]), variant),
                seed=seed_for(BASE, "init", 5, dev_seed))
        for method, pol in policies.items():
            key = dict(budget=budget, dev_seed=dev_seed, method=method,
                       fresh_seed=fresh_seed, fresh_paths=len(fresh))
            guard_rows.extend(dict(**key, **r) for r in guard_diagnostic(opt, fresh, pol))
            guarded = discounted_payoffs(opt, fresh, pol, True)[0]
            unguarded = discounted_payoffs(opt, fresh, pol, False)[0]
            gain, se = paired(guarded, unguarded)
            guard_policy_rows.append(dict(**key, guard_effect=gain, paired_se=se,
                                          ci_low=gain-1.96*se, ci_high=gain+1.96*se))
guard_df = pd.DataFrame(guard_rows)
guard_effect_df = pd.DataFrame(guard_policy_rows)
guard_df.to_csv(OUT / "development_guard_dates.csv", index=False)
guard_effect_df.to_csv(OUT / "development_guard_policy.csv", index=False)
display(guard_df.groupby(["method", "guarded"])[["validation_gain", "fresh_gain"]].mean())
display(guard_effect_df)
